In [1]:
import torch
import torch.nn as nn
from torch.profiler import profile, ProfilerActivity, record_function
import pandas as pd
# 1. Define a simple linear layer and dummy input data
linear_layer = nn.Linear(in_features=4096, out_features=4096)
x = torch.randn(1024, 4096)

# 2. Wrap the execution in the profiler context manager
with profile(
        activities=[ProfilerActivity.CPU],  # Switch to ProfilerActivity.CUDA if profiling on a GPU
        record_shapes=True,                # Records the sizes of input tensors
        profile_memory=True                # Tracks memory allocation and release
) as prof:

    with record_function("linear_forward"):
        # This is your simple one-line net.Linear method call
        y = linear_layer(x)

# 3. Print the performance summary table

#3. Extract metrics into a list of dictionaries
prof_data = []
for entry in prof.key_averages():
    prof_data.append({
        "Name": entry.key,
        "CPU Time (ms)": entry.cpu_time_total / 1000.0,      # Convert microseconds to ms
        "CPU Mem (MB)": entry.cpu_memory_usage / (1024**2),  # Convert bytes to MB
        "Input Shapes": str(entry.input_shapes),
        "Call Count": entry.count
    })

# 4. Create the pandas DataFrame
df = pd.DataFrame(prof_data)


USDT:2026-09-20 19:32:19 6864:339783 SyncActivityProfilerHandler.cpp:39] profiler_start
USDT:2026-09-20 19:32:19 6864:339783 SyncActivityProfilerHandler.cpp:46] profiler_stop


In [2]:
df_filtered = df[df["Name"] != "linear_forward"]


In [3]:
# Keep only official internal PyTorch ATen operators
df_aten_only = df[df["Name"].str.startswith("aten::")]


In [4]:
df_aten_only

,Name,CPU Time (ms),CPU Mem (MB),Input Shapes,Call Count
1,aten::linear,26.200096,16.0,,1
2,aten::t,0.028059,0.0,,1
3,aten::transpose,0.011166,0.0,,1
4,aten::as_strided,0.004041,0.0,,2
5,aten::addmm,26.155037,16.0,,1
6,aten::expand,0.001500,0.0,,1
7,aten::copy_,0.811262,0.0,,1
8,aten::resolve_conj,0.001291,0.0,,2


In [6]:
import torch
import torch.nn as nn

# 1. Setup simple linear operation
linear_layer = nn.Linear(in_features=4096, out_features=4096)
x = torch.randn(1024, 4096)

# 2. Run the profiler with the TensorBoard trace handler
with torch.profiler.profile(
        activities=[torch.profiler.ProfilerActivity.CPU], # Add .CUDA here if using a GPU
        schedule=torch.profiler.schedule(wait=1, warmup=1, active=3, repeat=1),
        on_trace_ready=torch.profiler.tensorboard_trace_handler('./log/linear_profile'),
        record_shapes=True,
        profile_memory=True
) as prof:

    # Run a few iterations to satisfy the scheduler (wait, warmup, active)
    for step in range(5):
        y = linear_layer(x)
        prof.step()  # Crucial: signals the profiler to advance steps


USDT:2026-09-20 19:39:28 6864:339783 SyncActivityProfilerHandler.cpp:39] profiler_start
USDT:2026-09-20 19:39:28 6864:339783 SyncActivityProfilerHandler.cpp:46] profiler_stop


In [7]:
!tensorboard --logdir=./log --bind_all

TensorFlow installation not found - running with reduced feature set.
I0920 19:39:41.880849 6124630016 plugin.py:429] Monitor runs begin
I0920 19:39:41.881244 6124630016 plugin.py:444] Find run directory /Users/deven/Developer/Machine_Learning_Algorithms/MIT/6_7960/weekly/week4/homeworks/log/linear_profile
I0920 19:39:41.881438 6158282752 plugin.py:493] Load run linear_profile
I0920 19:39:41.887591 6158282752 loader.py:57] started all processing
Serving TensorBoard on localhost; to expose to the network, use a proxy or pass --bind_all
TensorBoard 2.21.0 at http://localhost:6007/ (Press CTRL+C to quit)
I0920 19:39:42.268865 6158282752 plugin.py:497] Run linear_profile loaded
I0920 19:39:42.269221 6141456384 plugin.py:467] Add run linear_profile
W0920 19:39:45.502510 6175109120 application.py:559] path /data/index.js not found, sending 404
W0920 19:41:39.091173 6175109120 application.py:559] path /data/index.js not found, sending 404
^C


In [8]:
import torch
from torch.utils.tensorboard import SummaryWriter
writer = SummaryWriter()

In [13]:
import  torch.utils.tensorboard as tb

In [14]:
dir(tb)

['FileWriter',
 'RecordWriter',
 'SummaryWriter',
 '__all__',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__path__',
 '__spec__',
 '_convert_np',
 '_embedding',
 '_onnx_graph',
 '_proto_graph',
 '_pytorch_graph',
 '_utils',
 'summary',
 'writer']

In [16]:
writer1 = SummaryWriter()

In [9]:
x = torch.arange(-5, 5, 0.1).view(-1, 1)
y = -5 * x + 0.1 * torch.randn(x.size())

model = torch.nn.Linear(1, 1)
criterion = torch.nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr = 0.1)

def train_model(iter):
    for epoch in range(iter):
        y1 = model(x)
        loss = criterion(y1, y)
        writer.add_scalar("Loss/train", loss, epoch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

train_model(10)
writer.flush()

In [11]:
!tensorboard --logdir=runs


TensorFlow installation not found - running with reduced feature set.
I0920 19:45:04.209093 6165540864 plugin.py:429] Monitor runs begin
Serving TensorBoard on localhost; to expose to the network, use a proxy or pass --bind_all
TensorBoard 2.21.0 at http://localhost:6006/ (Press CTRL+C to quit)
^C


In [12]:
import torch
import torch.nn as nn

# 1. Setup simple linear operation
linear_layer = nn.Linear(in_features=4096, out_features=4096)
x = torch.randn(1024, 4096)

# 2. Run the profiler with the TensorBoard trace handler
with torch.profiler.profile(
        activities=[torch.profiler.ProfilerActivity.CPU], # Add .CUDA here if using a GPU
        schedule=torch.profiler.schedule(wait=1, warmup=1, active=3, repeat=1),
        on_trace_ready=torch.profiler.tensorboard_trace_handler('./logs'),
        record_shapes=True,
        profile_memory=True
) as prof:

    # Run a few iterations to satisfy the scheduler (wait, warmup, active)
    for step in range(5):
        y = linear_layer(x)
        prof.step()  # Crucial: signals the profiler to advance steps


USDT:2026-09-20 19:49:38 6864:339783 SyncActivityProfilerHandler.cpp:39] profiler_start
[W920 19:49:38.281591000 CPUAllocator.cpp:245] Memory block of unknown size was allocated before the profiling started, profiler results will not include the deallocation event
USDT:2026-09-20 19:49:38 6864:339783 SyncActivityProfilerHandler.cpp:46] profiler_stop


In [18]:
import torch
import torch.nn as nn
from torch.profiler import profile, record_function, ProfilerActivity, tensorboard_trace_handler

# 1. Setup a basic dummy model and input so the script runs standalone
dummy_model = nn.Linear(100, 100)
inp = torch.randn(10, 100)

# 2. WARMUP (CRITICAL)
# Run the model once outside the profiler to initialize memory and backend drivers.
# Without this, the profiler might capture nothing but framework initialization.
_ = dummy_model(inp)
if torch.backends.mps.is_available():
    torch.mps.synchronize()

# 3. START PROFILING
print("Starting profiler...")
with profile(
        activities=[ProfilerActivity.CPU],
        record_shapes=True,
        with_stack=True,
        # This automatically dumps the data to the folder when the 'with' block ends
        on_trace_ready=tensorboard_trace_handler('./runs/my_test_profile')
) as prof:

    with record_function("transformer_forward"):
        output = dummy_model(inp)

        if torch.backends.mps.is_available():
            torch.mps.synchronize()

print("Profiling complete! Check if './runs/my_test_profile' has a .pt.trace.json file.")

Starting profiler...
Profiling complete! Check if './runs/my_test_profile' has a .pt.trace.json file.


USDT:2026-09-20 20:16:15 6864:339783 SyncActivityProfilerHandler.cpp:39] profiler_start
USDT:2026-09-20 20:16:15 6864:339783 SyncActivityProfilerHandler.cpp:46] profiler_stop
